In [1]:
import torch
from astartes import train_test_split
import pandas as pd
import numpy as np

from chemprop.data import MoleculeDatapoint, MoleculeDataset, build_dataloader
from chemprop.featurizers import SimpleMoleculeMolGraphFeaturizer
from chemprop.models import MPNN
from chemprop.nn.transforms import UnscaleTransform


/home/jackson/miniforge3/envs/chemeleon-dev/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
featurizer = SimpleMoleculeMolGraphFeaturizer()

train_df = pd.read_csv("baybekov_ksol.csv")

smiles_col = "SMILES"
target = train_df[["logS"]].values

train_idxs, val_idxs = train_test_split(
    np.arange(train_df.shape[0]),
    train_size=0.80,
    test_size=0.20,
    random_state=42,
)
train_data = [
    MoleculeDatapoint.from_smi(smi, y)
    for smi, y in zip(
        train_df[smiles_col].iloc[train_idxs], target[train_idxs]
    )
]
val_data = [
    MoleculeDatapoint.from_smi(smi, y)
    for smi, y in zip(
        train_df[smiles_col].iloc[val_idxs], target[val_idxs]
    )
]

train_dataset = MoleculeDataset(train_data, featurizer)
val_dataset = MoleculeDataset(val_data, featurizer)
scaler = train_dataset.normalize_targets()
val_dataset.normalize_targets(scaler)
train_dataloader = build_dataloader(train_dataset, num_workers=1)
val_dataloader = build_dataloader(val_dataset, num_workers=1, shuffle=False)
output_transform = UnscaleTransform.from_standard_scaler(scaler)

model = MPNN.load_from_checkpoint(r"output/2025-10-01_13-49-22/checkpoints/epoch=8-step=5139.ckpt")


In [3]:
model.eval()
all_probs, all_assigns = [], []
with torch.no_grad():
    for batch in iter(val_dataloader):
        bmg = batch.bmg
        bmg.to(model.device)
        model.predictor(model.fingerprint(bmg))
        probs = model.predictor._last_gate_weights
        assigns = probs.argmax(dim=-1)
        all_probs.append(probs)
        all_assigns.append(assigns)
assigns = torch.cat(all_assigns)
probs = torch.cat(all_probs)

In [4]:
probs.softmax(axis=1).sum(axis=0)

tensor([1721.4719, 3301.0186, 2425.8516, 1679.6580], device='cuda:0')

In [5]:
hard_assign_counts = torch.bincount(assigns, minlength=model.predictor.n_experts).cpu()

In [6]:
hard_assign_counts

tensor([   0, 6560, 2568,    0])